# 08 Strategy gallery

Thirteen ready-made strategies run side by side on one instrument and one window through the
same `BacktestRunner` as `04_backtest_evaluation`: five of Nautilus's own example strategies by
string path (`EMACross`, `EMACrossLongOnly`, `EMACrossBracket`, `BBMeanReversion` and
`EMACrossTWAP` with its `TWAPExecAlgorithm`), three `MACrossStrategy` runs (a different moving
average and exit each) and five `IndicatorSignalStrategy` runs (Bollinger, MACD, RSI, OBV and the
fuzzy candle). The page shows them as a leaderboard and one overlaid equity chart, then repeats
one of them across every simulated-exchange model Nautilus offers (11 fill models, 3 fee models,
three latency settings) to show how much of a result is the strategy and how much is a modelling
choice.

Every number is a `RunResult` field (`research.application.gallery`): the statistics are
`MetricReport`'s, exactly `kernel.performance_metrics.all_metrics`, so no cell sums or averages a
PnL. A run that raises keeps its row with the error text in the `error` column -- a failure is
shown, never skipped.

`EMACrossTrailingStop` and `VolatilityMarketMaker` are not in the gallery: they read the latest
quote and the bars feed carries none, so they would never place an order here (the doc's
section 3).

## 1. Parameters

Read from the environment (`research/notebooks/_params.py`): `CATALOG_PATH`, `INSTRUMENTS`,
`START`, `END`. The rest are set here; each can also be given as JSON in `NOTEBOOK_<NAME>`
(`_params.setting`), which is how the test harness shrinks them to its ten-minute fixture:

- `INSTRUMENT` -- the instrument backtested, by default the first of `INSTRUMENTS`;
- `DATA` -- the feed; the gallery needs bars, `"bars:1-MINUTE"` (Nautilus aggregates them from
  the trade archive);
- `PERIODS` -- overrides of every period, multiple and threshold, `{}` for each strategy's own
  defaults. A key is generic (`fast`, `slow`, `atr_period`, `atr_multiple`, `k`, ...) or for one
  strategy (`"EMACross.fast"`, `"Signal.rsi.low"`, `"MACross.HULL.atr_multiple"`); the spec of
  each row shows what it resolved to;
- `STARTING_BALANCE` -- in the instrument's settlement currency;
- `AXIS_SPEC` -- the label of the gallery row §3 repeats across the execution models;
- `SEED` -- the random seed of every fill model, so a rerun is identical.

In [ ]:
import plotly.graph_objects as go
from _params import Params
from _params import setting
from research.application import gallery
from research.application.backtest_runner import NodeRunner


params = Params.from_env()
INSTRUMENT = setting("INSTRUMENT", params.instruments[0])
DATA = setting("DATA", "bars:1-MINUTE")
PERIODS = setting("PERIODS", {})
STARTING_BALANCE = setting("STARTING_BALANCE", 10_000)
AXIS_SPEC = setting("AXIS_SPEC", "EMACross")
SEED = setting("SEED", 42)
runner = NodeRunner()
specs = gallery.default_specs(
    params.catalog_path,
    INSTRUMENT,
    params.start,
    params.end,
    DATA,
    PERIODS,
    STARTING_BALANCE,
)
print(f"{INSTRUMENT}, {DATA}, {params.start} -> {params.end}: {len(specs)} strategies")
print(f"sizes {PERIODS or 'each strategy default'}; execution axes repeat {AXIS_SPEC!r}")

## 2. Leaderboard

Every spec run once (`gallery.run_specs`), one row each: the `MetricReport` statistics (win rate,
expectancy, Sharpe, Sortino, Calmar, max drawdown, profit factor, ... -- None where undefined, e.g.
a return statistic on a window shorter than two UTC days), the closed `trades`, the engine's
`orders` and `fills`, the engine's run time (`wall_seconds`), the whole call's time
(`total_seconds`, data loading included) and the `error`. The default order is the gallery's,
not a ranking; sort on the column you care about.

**The equity is the account balance:** realized PnL and every fee, not marked to market (the
`RunResult` Known limit), so a position held open at the end of the window shows only its entry
fee.

In [ ]:
outcomes = gallery.run_specs(runner, specs)
board = gallery.leaderboard(outcomes)
print(board.to_string(index=False))
gallery.check_outcomes(outcomes)  # the board shows a failed row first, then the run fails loudly
equity = gallery.equity_frame(outcomes)
equity_fig = go.Figure(
    [
        go.Scatter(x=rows["ts"], y=rows["equity"], name=label, line_shape="hv")
        for label, rows in equity.groupby("label")
    ]
)
equity_fig.update_layout(title=f"{INSTRUMENT}: account equity per strategy", height=560)
equity_fig.show()

## 3. Execution axes

The `AXIS_SPEC` row repeated with exactly one thing changed (`gallery.execution_axes`): each of
the 11 fill models (probabilities 0.5 / 0.5, seeded with `SEED`), each of the 3 fee models (a flat
0.1 commission for the fixed and per-contract ones, in the settlement currency) and three latency
settings (none, separate insert / update / cancel delays, and the baseline's own fixed 300 ms,
shown as the baseline row rather than run again). The tables are the
same as the leaderboard with the `axis` and `value` columns; the figure then plots, per setting,
how far each fill's price is from the same order's price in the baseline run (the gallery row,
i.e. the venue defaults and a 300 ms delay), in basis points (`gallery.slippage_by_axis`: orders
matched on id, side and quantity; an order the baseline did not fill is left out, not zero).

**Read it as sensitivity:** a result that survives every row is robust to the exchange model; one
that flips sign under a fee or a latency setting was never an edge. A fill model is a
hypothesis about the venue until it is measured (the doc's section 5): the table shows what the
hypotheses do, not what the venue does.

In [ ]:
baseline = gallery.find_outcome(outcomes, AXIS_SPEC)
axes = gallery.execution_axes(baseline.gallery, SEED)
axis_outcomes = gallery.run_specs(runner, axes)
axis_board = gallery.axis_table(axis_outcomes, baseline)
for axis in ("fill", "fee", "latency"):
    print(
        axis_board[axis_board["axis"] == axis]
        .drop(columns=["label", "axis", "strategy"])
        .to_string(index=False)
    )
differences = gallery.slippage_by_axis(axis_outcomes, baseline)
difference_fig = go.Figure(
    [
        go.Histogram(x=rows["difference_bps"], name=f"{axis}: {value}", opacity=0.6)
        for (axis, value), rows in differences.groupby(["axis", "value"])
    ]
)
difference_fig.update_layout(
    title=f"{AXIS_SPEC}: fill price against the baseline run (bps)", barmode="overlay"
)
difference_fig.show()
print(f"{len(differences)} matched fills across {len(axes)} settings")

## 4. Reading guide

- **Copy a row into `04_backtest_evaluation`.** A row is a `RunSpec`: in notebook 04 set
  `STRATEGY` and `STRATEGY_CONFIG` to the string paths of the row's strategy (an upstream example
  is `nautilus_trader.examples.strategies.<file>:<Class>`, a family strategy is
  `research.strategies.ma_cross_strategy:MACrossStrategy` or
  `research.strategies.indicator_signal_strategy:IndicatorSignalStrategy`, each with its
  `...Config`), `PARAMS` to the row's parameters (`print(specs[i].spec)` shows them, resolved from
  `PERIODS`), `DATA` to `"bars:1-MINUTE"`; then the single run, the sweep and the walk-forward
  judge it.
- **Everything here is in-sample.** Fixed, untuned parameters on one window: the leaderboard says
  what each strategy did on this stretch of data, not what it will do. A top row over a few
  trades is luck until a longer window, a sweep and a walk-forward say otherwise.
- **Few trades, short windows.** The return statistics need realized PnL on two UTC days; read
  `trades` before any rate. An empty column is an undefined statistic, never a zero.
- **A fill model is a hypothesis.** The fill, fee and latency rows change the simulated exchange,
  not the market: until the venue's real slippage, fees and order round trip are measured on the
  VPS, treat the spread between rows as the uncertainty of the result. Identical fill-model rows
  are the same modelling choice at this order size, not independent hypotheses: at `trade_size`
  0.01 on an L1 venue the tiered and size-aware models never reach a second tier (the catalog's
  section 5, pitfall 6).
- **One strategy per row, one position at a time.** Sizes are fixed (`trade_size` 0.01), there
  is no portfolio sizing, and the family strategies hold at most one position.